In [ ]:
# =============================================================================
# MULTIPLE MYELOMA STAGE PREDICTION — MODELING
# =============================================================================
# This notebook trains and evaluates multiple classification models to predict
# the clinical stage of Multiple Myeloma patients using the Durie-Salmon
# staging system. Experiment tracking is done with MLflow.
#
# Target: 3-class problem (Stage I, Stage II, Stage III)
# Features: clinical biomarkers from the MM-dataset (Tlemcen, Algeria)
# Models: Random Forest, XGBoost
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from xgboost import XGBClassifier

print("Libraries loaded OK")

In [ ]:
# import mlflow

# Set MLflow tracking URI to project root
mlflow.set_tracking_uri("file:///home/miguel/myeloma-explainability/mlruns")
mlflow.set_experiment("mm_stage_prediction")

In [ ]:
# Load processed dataset
df_model_raw = pd.read_csv('../data/mm_processed.csv')
print(f"Loaded processed dataset: {df_model_raw.shape}")

In [ ]:
# =============================================================================
# TARGET PREPARATION
# =============================================================================
# Create 3-class target based on Durie-Salmon staging system.
# Exclude MGUS, ASYM and PLASMO due to insufficient sample size.
# =============================================================================

# Filter and map to 3-class target
stage_mapping = {
    'IA': 'Stage_I', 'IB': 'Stage_I',
    'IIA': 'Stage_II', 'IIB': 'Stage_II',
    'IIIA': 'Stage_III', 'IIIB': 'Stage_III'
}

df_model = df_model_raw[df_model_raw['CLASS_LABEL'].isin(stage_mapping.keys())].copy()
df_model['STAGE'] = df_model['CLASS_LABEL'].map(stage_mapping)

print(f"Dataset for modeling: {df_model.shape}")
print(f"\nTarget distribution:")
print(df_model['STAGE'].value_counts())

In [ ]:
# =============================================================================
# FEATURE PREPARATION AND TRAIN/TEST SPLIT
# =============================================================================
# Encoding categorical features and preparing X, y for modeling.
# Stratified split to preserve class distribution given imbalanced dataset.
# =============================================================================

# Drop non-feature columns
drop_cols = ['CLASS', 'CLASS_LABEL', 'STAGE']
feature_cols = [c for c in df_model.columns if c not in drop_cols]

# Encode categorical features
df_encoded = df_model.copy()
categorical_cols = df_encoded[feature_cols].select_dtypes(include='object').columns
print(f"Categorical features to encode: {list(categorical_cols)}")

le = LabelEncoder()
for col in categorical_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

# Prepare X and y
X = df_encoded[feature_cols]
y = df_encoded['STAGE']

# Encode target
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)
print(f"\nTarget classes: {le_target.classes_}")

# Train/test split — stratified to preserve class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")
print(f"\nClass distribution in test set:")
for i, cls in enumerate(le_target.classes_):
    print(f"  {cls}: {(y_test == i).sum()} patients")

In [ ]:
# =============================================================================
# MLFLOW SETUP AND MODEL TRAINING
# =============================================================================
# Training Random Forest and XGBoost with MLflow experiment tracking.
# Using class_weight/scale_pos_weight to handle class imbalance.
# Evaluation metric: weighted F1-score (appropriate for imbalanced datasets)
# =============================================================================

mlflow.set_experiment("mm_stage_prediction")

def evaluate_model(model, X_test, y_test, le_target):
    """Evaluate model and return metrics and predictions."""
    y_pred = model.predict(X_test)
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    f1_macro = f1_score(y_test, y_pred, average='macro')
    report = classification_report(y_test, y_pred, 
                                   target_names=le_target.classes_,
                                   output_dict=True)
    return y_pred, f1_weighted, f1_macro, report

# --- Model 1: Random Forest ---
with mlflow.start_run(run_name="RandomForest_baseline"):
    rf = RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42
    )
    rf.fit(X_train, y_train)
    y_pred_rf, f1_w_rf, f1_m_rf, report_rf = evaluate_model(rf, X_test, y_test, le_target)
    
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("f1_weighted", f1_w_rf)
    mlflow.log_metric("f1_macro", f1_m_rf)
    mlflow.sklearn.log_model(rf, "model")
    
    print(f"Random Forest — F1 weighted: {f1_w_rf:.3f} | F1 macro: {f1_m_rf:.3f}")

# --- Model 2: XGBoost ---
with mlflow.start_run(run_name="XGBoost_baseline"):
    xgb = XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric='mlogloss',
        verbosity=0
    )
    xgb.fit(X_train, y_train)
    y_pred_xgb, f1_w_xgb, f1_m_xgb, report_xgb = evaluate_model(xgb, X_test, y_test, le_target)
    
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_metric("f1_weighted", f1_w_xgb)
    mlflow.log_metric("f1_macro", f1_m_xgb)
    mlflow.sklearn.log_model(xgb, "model")
    
    print(f"XGBoost       — F1 weighted: {f1_w_xgb:.3f} | F1 macro: {f1_m_xgb:.3f}")

print("\nBoth models logged to MLflow ✅")

In [ ]:
# =============================================================================
# MODEL EVALUATION — CONFUSION MATRICES AND CLASSIFICATION REPORTS
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title in zip(axes,
                              [y_pred_rf, y_pred_xgb],
                              ['Random Forest', 'XGBoost']):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
                xticklabels=le_target.classes_,
                yticklabels=le_target.classes_,
                ax=ax)
    ax.set_title(f'{title}\nF1 weighted: {f1_score(y_test, y_pred, average="weighted"):.3f} | F1 macro: {f1_score(y_test, y_pred, average="macro"):.3f}',
                 fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — Baseline Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../notebooks/confusion_matrices_baseline.png', dpi=150)
plt.show()

# Detailed classification report
print("=== RANDOM FOREST — CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred_rf, target_names=le_target.classes_))

print("=== XGBOOST — CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred_xgb, target_names=le_target.classes_))

In [ ]:
# =============================================================================
# HANDLING CLASS IMBALANCE WITH SMOTE
# =============================================================================
# SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic
# samples for minority classes by interpolating between existing samples.
# Applied ONLY on training data — never on test data.
#
# Original distribution: Stage_III=145, Stage_I=25, Stage_II=20
# After SMOTE: balanced classes
# =============================================================================

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("=== CLASS DISTRIBUTION AFTER SMOTE ===")
for i, cls in enumerate(le_target.classes_):
    original = (y_train == i).sum()
    resampled = (y_train_smote == i).sum()
    print(f"  {cls}: {original} → {resampled} samples")

print(f"\nTraining set shape: {X_train.shape} → {X_train_smote.shape}")

In [ ]:
# =============================================================================
# MODEL TRAINING WITH SMOTE-BALANCED DATA
# Full MLflow logging: parameters, metrics per class, dataset, plots
# =============================================================================

# --- Model 1: Random Forest with SMOTE (best model — full logging) ---
with mlflow.start_run(run_name="RandomForest_SMOTE"):

    rf_smote = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_smote.fit(X_train_smote, y_train_smote)
    y_pred_rf_smote, f1_w_rf_smote, f1_m_rf_smote, report_rf_smote = evaluate_model(
        rf_smote, X_test, y_test, le_target)

    # Parameters
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("smote", True)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("missing_threshold_drop", 50)
    mlflow.log_param("imputation_strategy", "median")

    # Dataset info
    mlflow.log_param("dataset_total_patients", len(df))
    mlflow.log_param("dataset_modeling_patients", len(df_model))
    mlflow.log_param("dataset_train_size", len(X_train_smote))
    mlflow.log_param("dataset_test_size", len(X_test))
    mlflow.log_param("dataset_features", X.shape[1])
    mlflow.log_param("dataset_source", "MM-dataset Mendeley 7wpcv7kp6f")

    # Global metrics
    mlflow.log_metric("f1_weighted", f1_w_rf_smote)
    mlflow.log_metric("f1_macro", f1_m_rf_smote)

    # Metrics per class
    for cls in le_target.classes_:
        mlflow.log_metric(f"f1_{cls}", report_rf_smote[cls]['f1-score'])
        mlflow.log_metric(f"precision_{cls}", report_rf_smote[cls]['precision'])
        mlflow.log_metric(f"recall_{cls}", report_rf_smote[cls]['recall'])

    # Model
    mlflow.sklearn.log_model(rf_smote, "model")

    # Dataset artifact
    mlflow.log_artifact("../data/mm_processed.csv", artifact_path="dataset")

    # Confusion matrix
    fig, ax = plt.subplots(figsize=(7, 5))
    cm = confusion_matrix(y_test, y_pred_rf_smote)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
                xticklabels=le_target.classes_,
                yticklabels=le_target.classes_, ax=ax)
    ax.set_title('Confusion Matrix — RF + SMOTE', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    cm_path = '../notebooks/confusion_matrix_best_model.png'
    plt.savefig(cm_path, dpi=150)
    plt.close()
    mlflow.log_artifact(cm_path, artifact_path="plots")

    # EDA plots
    mlflow.log_artifact('../notebooks/class_distribution.png', artifact_path="plots")
    mlflow.log_artifact('../notebooks/missing_values.png', artifact_path="plots")
    mlflow.log_artifact('../notebooks/models_comparison.png', artifact_path="plots")
    mlflow.log_artifact('../notebooks/durie_salmon_validation.png', artifact_path="plots")

    print(f"Random Forest + SMOTE — F1 weighted: {f1_w_rf_smote:.3f} | F1 macro: {f1_m_rf_smote:.3f}")
    print("\nLogged to MLflow:")
    print("  ✅ Parameters + dataset info")
    print("  ✅ Global metrics + metrics per class")
    print("  ✅ Model")
    print("  ✅ Dataset artifact")
    print("  ✅ Plots")

# --- Model 2: XGBoost with SMOTE ---
with mlflow.start_run(run_name="XGBoost_SMOTE"):
    xgb_smote = XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric='mlogloss',
        verbosity=0
    )
    xgb_smote.fit(X_train_smote, y_train_smote)
    y_pred_xgb_smote, f1_w_xgb_smote, f1_m_xgb_smote, report_xgb_smote = evaluate_model(
        xgb_smote, X_test, y_test, le_target)

    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("smote", True)
    mlflow.log_metric("f1_weighted", f1_w_xgb_smote)
    mlflow.log_metric("f1_macro", f1_m_xgb_smote)
    for cls in le_target.classes_:
        mlflow.log_metric(f"f1_{cls}", report_xgb_smote[cls]['f1-score'])
    mlflow.sklearn.log_model(xgb_smote, "model")

    print(f"XGBoost + SMOTE       — F1 weighted: {f1_w_xgb_smote:.3f} | F1 macro: {f1_m_xgb_smote:.3f}")

print("\nBoth SMOTE models logged to MLflow ✅")

In [ ]:
# =============================================================================
# EVALUATION — SMOTE MODELS
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title in zip(axes,
                              [y_pred_rf_smote, y_pred_xgb_smote],
                              ['Random Forest + SMOTE', 'XGBoost + SMOTE']):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
                xticklabels=le_target.classes_,
                yticklabels=le_target.classes_,
                ax=ax)
    ax.set_title(f'{title}\nF1 weighted: {f1_score(y_test, y_pred, average="weighted"):.3f} | F1 macro: {f1_score(y_test, y_pred, average="macro"):.3f}',
                 fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — SMOTE Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../notebooks/confusion_matrices_smote.png', dpi=150)
plt.show()

print("=== RANDOM FOREST + SMOTE — CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred_rf_smote, target_names=le_target.classes_))

print("=== XGBOOST + SMOTE — CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred_xgb_smote, target_names=le_target.classes_))

In [ ]:
# =============================================================================
# HYPERPARAMETER TUNING — RANDOM FOREST
# =============================================================================
# Grid search with cross-validation on SMOTE-balanced training data.
# Using StratifiedKFold to preserve class distribution in each fold.
# =============================================================================

from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

with mlflow.start_run(run_name="RandomForest_GridSearch"):
    grid_search = GridSearchCV(
        RandomForestClassifier(random_state=42),
        param_grid,
        cv=cv,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=1
    )
    grid_search.fit(X_train_smote, y_train_smote)
    
    best_rf = grid_search.best_estimator_
    y_pred_best_rf, f1_w_best, f1_m_best, _ = evaluate_model(
        best_rf, X_test, y_test, le_target)
    
    mlflow.log_params(grid_search.best_params_)
    mlflow.log_param("smote", True)
    mlflow.log_metric("f1_weighted", f1_w_best)
    mlflow.log_metric("f1_macro", f1_m_best)
    mlflow.sklearn.log_model(best_rf, "model")
    
    print(f"Best params: {grid_search.best_params_}")
    print(f"Best RF — F1 weighted: {f1_w_best:.3f} | F1 macro: {f1_m_best:.3f}")

In [ ]:
# =============================================================================
# MODELS COMPARISON SUMMARY
# =============================================================================

results = {
    'Model': [
        'RF baseline', 
        'XGBoost baseline',
        'RF + SMOTE',
        'XGBoost + SMOTE',
        'RF + SMOTE + GridSearch'
    ],
    'F1 weighted': [f1_w_rf, f1_w_xgb, f1_w_rf_smote, f1_w_xgb_smote, f1_w_best],
    'F1 macro':    [f1_m_rf, f1_m_xgb, f1_m_rf_smote, f1_m_xgb_smote, f1_m_best]
}

df_results = pd.DataFrame(results).sort_values('F1 macro', ascending=False)
print("=== MODELS COMPARISON ===\n")
print(df_results.to_string(index=False))
print(f"\n✅ Best model: RF + SMOTE (F1 macro: {f1_m_rf_smote:.3f})")

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(df_results))
ax.bar([i - 0.2 for i in x], df_results['F1 weighted'], 
       width=0.4, label='F1 weighted', color='#8e44ad', alpha=0.8)
ax.bar([i + 0.2 for i in x], df_results['F1 macro'],    
       width=0.4, label='F1 macro', color='#f9ca24', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(df_results['Model'], rotation=15, ha='right')
ax.set_ylabel('F1 Score')
ax.set_title('Models Comparison — F1 Scores', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../notebooks/models_comparison.png', dpi=150)
plt.show()